# Business Central → Fabric Warehouse: Bronze Layer Ingestion

This notebook pulls data from Business Central via its REST API and lands it in the **bronze** schema of your Fabric Warehouse — raw, as-is, with an ingestion timestamp appended.

**Tables loaded:**
- `bronze.customer`
- `bronze.item`
- `bronze.sales_order_header`
- `bronze.sales_order_line`

**Prerequisites:**
1. A Fabric Workspace with a Warehouse created.
2. Business Central credentials stored as Notebook environment variables or Azure Key Vault secrets.
3. The Warehouse SQL connection string (find it in the Warehouse settings → SQL connection string).

## 1. Install dependencies

In [ ]:
%pip install requests pyodbc azure-identity --quiet

## 2. Configuration — fill in your values

In [ ]:
import os

# ── Business Central ──────────────────────────────────────────────────────────
BC_TENANT_ID     = os.environ.get("BC_TENANT_ID",     "<your-aad-tenant-id>")
BC_CLIENT_ID     = os.environ.get("BC_CLIENT_ID",     "<your-app-client-id>")
BC_CLIENT_SECRET = os.environ.get("BC_CLIENT_SECRET", "<your-app-client-secret>")
BC_ENVIRONMENT   = os.environ.get("BC_ENVIRONMENT",   "Production")   # or 'Sandbox'
BC_COMPANY_ID    = os.environ.get("BC_COMPANY_ID",    "<your-bc-company-id>")

BC_BASE_URL = (
    f"https://api.businesscentral.dynamics.com/v2.0/{BC_TENANT_ID}"
    f"/{BC_ENVIRONMENT}/api/v2.0/companies({BC_COMPANY_ID})"
)

# ── Fabric Warehouse (ODBC / SQL endpoint) ────────────────────────────────────
# Found in: Fabric Workspace → Warehouse → Settings → SQL connection string
FABRIC_SERVER = os.environ.get("FABRIC_SERVER", "<workspace>.datawarehouse.fabric.microsoft.com")
FABRIC_DATABASE = os.environ.get("FABRIC_DATABASE", "<your-warehouse-name>")

print("Config loaded.")

## 3. Authenticate with Business Central (OAuth 2.0 client credentials)

In [ ]:
import requests

def get_bc_token() -> str:
    """Obtain a bearer token for the BC API using client-credentials flow."""
    url = f"https://login.microsoftonline.com/{BC_TENANT_ID}/oauth2/v2.0/token"
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     BC_CLIENT_ID,
        "client_secret": BC_CLIENT_SECRET,
        "scope":         "https://api.businesscentral.dynamics.com/.default",
    }
    resp = requests.post(url, data=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()["access_token"]


BC_TOKEN = get_bc_token()
print("BC token acquired.")

## 4. Generic BC API fetcher (handles OData pagination)

In [ ]:
from typing import List, Dict, Any

def fetch_bc_entity(endpoint: str, select: str = "", top: int = 5000) -> List[Dict[str, Any]]:
    """Page through a BC OData endpoint and return all records."""
    url = f"{BC_BASE_URL}/{endpoint}"
    params: Dict[str, Any] = {"$top": top}
    if select:
        params["$select"] = select

    headers = {"Authorization": f"Bearer {BC_TOKEN}", "Accept": "application/json"}
    records: List[Dict[str, Any]] = []

    while url:
        resp = requests.get(url, headers=headers, params=params, timeout=60)
        resp.raise_for_status()
        body = resp.json()
        records.extend(body.get("value", []))
        url = body.get("@odata.nextLink")  # None when last page
        params = {}  # nextLink already contains all params

    return records


print("Fetcher defined.")

## 5. Pull BC data into Spark DataFrames

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timezone

spark = SparkSession.builder.getOrCreate()
INGESTED_AT = datetime.now(timezone.utc).isoformat()


def bc_to_spark(endpoint: str, select: str = ""):
    """Fetch a BC entity and return a Spark DataFrame with _ingested_at appended."""
    rows = fetch_bc_entity(endpoint, select=select)
    df = spark.createDataFrame(rows)
    df = df.withColumn("_ingested_at", F.lit(INGESTED_AT).cast("timestamp"))
    df = df.withColumn("_source_system", F.lit("BusinessCentral"))
    return df


# ── Customer ──────────────────────────────────────────────────────────────────
df_customer = bc_to_spark(
    "customers",
    select="id,number,displayName,addressLine1,addressLine2,city,state,"
           "country,postalCode,currencyCode,phoneNumber,email,blocked,"
           "creditLimit,balance,lastModifiedDateTime",
)
print(f"Customers:         {df_customer.count():,} rows")

# ── Item ──────────────────────────────────────────────────────────────────────
df_item = bc_to_spark(
    "items",
    select="id,number,displayName,type,itemCategoryCode,unitPrice,unitCost,"
           "baseUnitOfMeasureCode,inventory,blocked,lastModifiedDateTime",
)
print(f"Items:             {df_item.count():,} rows")

# ── Sales Order Header ────────────────────────────────────────────────────────
df_header = bc_to_spark(
    "salesOrders",
    select="id,number,orderDate,postingDate,customerId,customerNumber,"
           "customerName,currencyCode,salesperson,status,"
           "totalAmountExcludingTax,totalTaxAmount,totalAmountIncludingTax,"
           "requestedDeliveryDate,shipToName,shipToAddressLine1,shipToCity,"
           "shipToCountry,lastModifiedDateTime",
)
print(f"Sales order headers: {df_header.count():,} rows")

# ── Sales Order Line ──────────────────────────────────────────────────────────
df_line = bc_to_spark(
    "salesOrderLines",
    select="id,documentId,sequence,itemId,itemNumber,description,"
           "unitOfMeasureCode,quantity,unitPrice,discountAmount,discountPercent,"
           "taxPercent,lineAmount,shipmentDate,lastModifiedDateTime",
)
print(f"Sales order lines:   {df_line.count():,} rows")

## 6. Write to Fabric Warehouse — Bronze schema

We use the Fabric Spark connector (`com.microsoft.sqlserver.jdbc.spark`) to write directly to the SQL Warehouse endpoint.  
The **bronze** schema must exist in the warehouse before running this cell (`CREATE SCHEMA bronze;`).

In [ ]:
JDBC_URL = (
    f"jdbc:sqlserver://{FABRIC_SERVER}:1433;"
    f"database={FABRIC_DATABASE};"
    "authentication=ActiveDirectoryServicePrincipal;"
    f"AADSecurePrincipalId={BC_CLIENT_ID};"
    f"AADSecurePrincipalSecret={BC_CLIENT_SECRET};"
    f"AADTenantId={BC_TENANT_ID};"
    "encrypt=true;trustServerCertificate=false;"
    "loginTimeout=30;"
)

JDBC_OPTS = {
    "url":          JDBC_URL,
    "driver":       "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "batchsize":    "10000",
    "numPartitions": "8",
}


def write_bronze(df, table_name: str):
    """Overwrite a bronze table in the Fabric Warehouse."""
    print(f"  Writing bronze.{table_name} ...", end=" ")
    (
        df.write.format("jdbc")
        .options(**JDBC_OPTS)
        .option("dbtable", f"bronze.{table_name}")
        .mode("overwrite")   # full-refresh; swap for 'append' + dedup in dbt for incremental
        .save()
    )
    print("done.")


write_bronze(df_customer, "customer")
write_bronze(df_item,     "item")
write_bronze(df_header,   "sales_order_header")
write_bronze(df_line,     "sales_order_line")

print("\nAll bronze tables refreshed.")

## 7. (Optional) Trigger dbt run from the notebook

If you have the dbt CLI installed in the cluster environment you can kick off the downstream transformations directly after ingestion.

In [ ]:
import subprocess, sys

DBT_PROJECT_DIR = "/path/to/dbt_bc_project"   # adjust to your workspace path
DBT_PROFILES_DIR = DBT_PROJECT_DIR

result = subprocess.run(
    [
        sys.executable, "-m", "dbt", "run",
        "--project-dir", DBT_PROJECT_DIR,
        "--profiles-dir", DBT_PROFILES_DIR,
        "--target", "fabric",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)
    raise RuntimeError("dbt run failed — see stderr above.")